# Permutation CSV 文件准备

从原始 Mplus 结果 CSV 生成 6 组分析所需的 3 个 CSV 文件。

| # | group_id | 筛选条件 | tail | 含义 | 使用的 CSV |
|---|----------|----------|------|------|-----------|
| 1 | a_tail1 | 无 | 1 | 正的 a 路径效应 | `_a.csv` |
| 2 | a_tail-1 | 无 | -1 | 负的 a 路径效应（分散学习带来更多激活） | `_a.csv` |
| 3 | ab_a_gt_0_tail1 | 保留 a>0 | 1 | 正的间接效应（分散学习相比集中学习有更多理解加成） | `_ab_a大于0.csv` |
| 4 | ab_a_le_0_tail1 | 保留 a≤0 | 1 | 正的间接效应（a≤0 子集） | `_ab_a小于等于0.csv` |
| 5 | ab_a_gt_0_tail-1 | 保留 a>0 | -1 | 负的间接效应（a>0 子集） | `_ab_a大于0.csv` |
| 6 | ab_a_le_0_tail-1 | 保留 a≤0 | -1 | 负的间接效应（集中学习相比分散学习有更多理解加成） | `_ab_a小于等于0.csv` |

> 实际只需生成 3 个 CSV：`_a.csv`、`_ab_a大于0.csv`、`_ab_a小于等于0.csv`。
> 6 组的区别在于使用哪个 CSV + cluster finding 的 tail 参数（正/负方向）。
> 
> **a 路径**：脑活动条件差异（M-S），a<0 表示分散学习带来更多激活。
> **ab 间接效应**：脑活动通过行为中介影响学习效果，ab>0 表示分散学习相比集中学习有更多的理解加成。

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
from scipy import stats

## 函数定义

In [ ]:
def add_standard_stats_columns(csv_file_path, effect_type='ab'):
    """
    Add standard statistical columns (coef_estimate, std_error, t_value, p_value)
    based on selected effect type (a, b, or ab).

    Parameters:
    - csv_file_path: str, path to the CSV file containing mediation results
    - effect_type: str, 'a', 'b', or 'ab' to select which effect to analyze

    Returns:
    - output_path: str, path to the saved CSV file
    """
    data = pd.read_csv(csv_file_path)

    column_mapping = {
        'a': {'estimate': 'a_estimate_dat', 'std_error': 'a_std_error_dat'},
        'b': {'estimate': 'b_estimate_dat', 'std_error': 'b_std_error_dat'},
        'ab': {'estimate': 'ab_estimate_dat', 'std_error': 'ab_std_error_dat'}
    }

    if effect_type not in column_mapping:
        raise ValueError("effect_type must be 'a', 'b', or 'ab'")

    estimate_col = column_mapping[effect_type]['estimate']
    std_error_col = column_mapping[effect_type]['std_error']

    data['coef_estimate'] = data[estimate_col]
    data['std_error'] = data[std_error_col]

    with np.errstate(divide='ignore', invalid='ignore'):
        data['t_value'] = data['coef_estimate'] / data['std_error']
        data['t_value'] = data['t_value'].replace([np.inf, -np.inf], np.nan)

    data['p_value'] = 2 * (1 - stats.norm.cdf(np.abs(data['t_value'])))
    data['p_value'] = data['p_value'].fillna(1.0)

    base_name, ext = os.path.splitext(csv_file_path)
    output_path = f"{base_name}_{effect_type}{ext}"

    data.to_csv(output_path, index=False)
    print(f"  [{effect_type}] coef_estimate <- {estimate_col}, saved to: {os.path.basename(output_path)}")

    return output_path

In [ ]:
def split_csv_by_a_estimate(csv_file_path):
    """
    Split ab-path CSV into two based on a_estimate_dat sign.
    
    - _ab_a大于0.csv: 保留 a>0 的效应 (a<=0 的行 t_value=0)
    - _ab_a小于等于0.csv: 保留 a<=0 的效应 (a>0 的行 t_value=0)
    
    Returns: (output_path_keep_positive, output_path_keep_negative)
    """
    data = pd.read_csv(csv_file_path)

    # 使用已有的 coef_estimate/std_error/t_value/p_value 列
    # 如果没有，则从 ab 列计算
    if 'coef_estimate' not in data.columns:
        data['coef_estimate'] = data['ab_estimate_dat']
        data['std_error'] = data['ab_std_error_dat']
        with np.errstate(divide='ignore', invalid='ignore'):
            data['t_value'] = data['coef_estimate'] / data['std_error']
            data['t_value'] = data['t_value'].replace([np.inf, -np.inf], np.nan)
        data['p_value'] = 2 * (1 - stats.norm.cdf(np.abs(data['t_value'])))
        data['p_value'] = data['p_value'].fillna(1.0)

    # a<=0 的效应: a>0 的行被置零
    data_keep_negative = data.copy()
    mask_positive = data_keep_negative['a_estimate_dat'] >= 0
    data_keep_negative.loc[mask_positive, 't_value'] = 0
    data_keep_negative.loc[mask_positive, 'coef_estimate'] = 0
    data_keep_negative.loc[mask_positive, 'p_value'] = 1.0

    # a>0 的效应: a<=0 的行被置零
    data_keep_positive = data.copy()
    mask_negative = data_keep_positive['a_estimate_dat'] <= 0
    data_keep_positive.loc[mask_negative, 't_value'] = 0
    data_keep_positive.loc[mask_negative, 'coef_estimate'] = 0
    data_keep_positive.loc[mask_negative, 'p_value'] = 1.0

    base_name, ext = os.path.splitext(csv_file_path)
    output_path_keep_negative = f"{base_name}_a小于等于0{ext}"
    output_path_keep_positive = f"{base_name}_a大于0{ext}"

    data_keep_negative.to_csv(output_path_keep_negative, index=False)
    data_keep_positive.to_csv(output_path_keep_positive, index=False)

    print(f"  [ab split] {mask_positive.sum()} rows (a>=0) zeroed -> {os.path.basename(output_path_keep_negative)}")
    print(f"  [ab split] {mask_negative.sum()} rows (a<=0) zeroed -> {os.path.basename(output_path_keep_positive)}")

    return output_path_keep_positive, output_path_keep_negative

In [ ]:
def prepare_csv_for_groups(raw_csv_path):
    """
    从原始 Mplus 结果 CSV 生成 6 组分析所需的 3 个 CSV 文件。

    输入: 原始 CSV（含 a_estimate_dat, ab_estimate_dat 等列）
    
    生成文件:
      {base}_a.csv             — a 路径 (groups 1&2, 正负方向靠 tail 参数)
      {base}_ab.csv            — ab 间接效应 (中间产物)
      {base}_ab_a大于0.csv      — ab, 限 a>0 (groups 3&5, 正负方向靠 tail 参数)
      {base}_ab_a小于等于0.csv   — ab, 限 a≤0 (groups 4&6, 正负方向靠 tail 参数)

    Returns: dict mapping csv_key -> file_path
    """
    print(f"Processing: {os.path.basename(raw_csv_path)}")
    print(f"  Directory: {os.path.dirname(raw_csv_path)}")

    # Step 1: 生成 _a.csv 和 _ab.csv
    a_csv = add_standard_stats_columns(raw_csv_path, effect_type='a')
    ab_csv = add_standard_stats_columns(raw_csv_path, effect_type='ab')

    # Step 2: 拆分 ab 路径 -> _ab_a大于0.csv, _ab_a小于等于0.csv
    ab_pos_csv, ab_neg_csv = split_csv_by_a_estimate(ab_csv)

    csv_map = {
        '_a': a_csv,
        '_ab_a大于0': ab_pos_csv,
        '_ab_a小于等于0': ab_neg_csv,
    }

    print(f"  Done. Generated {len(csv_map)} CSV files.")
    return csv_map

## Real data 调用 + 验证

In [ ]:
BASE_CSV = r"D:\LYW\R for permutation t\Mplus\results\all_spatiotemporal_results_WLSMV_precise_parallel.csv"

csv_map = prepare_csv_for_groups(BASE_CSV)

# 验证
print("\n=== 验证 ===")
for key, path in csv_map.items():
    df = pd.read_csv(path)
    n_zero_t = (df['t_value'] == 0).sum()
    print(f"{key}: {len(df)} rows, t_value range [{df['t_value'].min():.3f}, {df['t_value'].max():.3f}], t=0 count: {n_zero_t}")

## Permutation 测试 (permutation_0001)

In [ ]:
permu_csv = r"D:\LYW\R for permutation t\Mplus\results\permutation_0001\all_spatiotemporal_results_WLSMV_precise_parallel.csv"

csv_map_permu = prepare_csv_for_groups(permu_csv)

# 验证
print("\n=== 验证 ===")
for key, path in csv_map_permu.items():
    df = pd.read_csv(path)
    n_zero_t = (df['t_value'] == 0).sum()
    print(f"{key}: {len(df)} rows, t_value range [{df['t_value'].min():.3f}, {df['t_value'].max():.3f}], t=0 count: {n_zero_t}")

## 批量处理所有 permutation 目录

In [ ]:
PERMU_BASE_DIR = r"D:\LYW\R for permutation t\Mplus\results"
permu_dirs = sorted(glob.glob(os.path.join(PERMU_BASE_DIR, "permutation_[0-9]*")))
print(f"Found {len(permu_dirs)} permutation directories")

for i, permu_dir in enumerate(permu_dirs):
    raw_csv = os.path.join(permu_dir, "all_spatiotemporal_results_WLSMV_precise_parallel.csv")
    if not os.path.exists(raw_csv):
        print(f"  SKIP {os.path.basename(permu_dir)}: CSV not found")
        continue
    
    print(f"\n[{i+1}/{len(permu_dirs)}] {os.path.basename(permu_dir)}")
    prepare_csv_for_groups(raw_csv)